# Apriori Algorithm Implementation Assignment

### Objective:
You will implement the **Apriori algorithm** from scratch (i.e., without using any libraries like `mlxtend`) to find frequent itemsets and generate association rules.

### Dataset:
Use the [Online Retail Dataset](https://www.kaggle.com/datasets/vijayuv/onlineretail) from Kaggle. You can filter it for a specific country (e.g., `United Kingdom`) and time range to reduce size if needed.

---

## Step 1: Data Preprocessing

- Load the dataset
- Remove rows with missing values
- Filter out rows where `Quantity <= 0`
- Convert Data into Basket Format

👉 **Implement code below**

In [22]:
# Load the dataset
# Preprocess as per the instructions above | We have already done in TASK 2
import pandas as pd
data = pd.read_csv('OnlineRetail.csv', encoding="ISO-8859-1")
data = data.dropna()
data = data[ data['Quantity'] >= 0]
data = data[data['Country'] == 'United Kingdom']

# Your Code Here
# for Basket
basket = data.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().reset_index().fillna(0).set_index('InvoiceNo')
basket = basket.map(lambda x: 1 if x > 0 else 0)
basket

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TOADSTOOL BEDSIDE LIGHT,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581582,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581583,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581584,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Step 2: Implement Apriori Algorithm
Step-by-Step Procedure:
1. Generate Frequent 1-Itemsets
Count the frequency (support) of each individual item in the dataset.
Keep only those with support ≥ min_support.
→ Result is L1 (frequent 1-itemsets)
2. Iterative Candidate Generation (k = 2 to n)
While L(k-1) is not empty:
a. Candidate Generation

Generate candidate itemsets Ck of size k from L(k-1) using the Apriori property:
Any (k-itemset) is only frequent if all of its (k−1)-subsets are frequent.
b. Prune Candidates
Eliminate candidates that have any (k−1)-subset not in L(k-1).
c. Count Support
For each transaction, count how many times each candidate in Ck appears.
d. Generate Frequent Itemsets
Form Lk by keeping candidates from Ck that meet the min_support.
Repeat until Lk becomes empty.
Implement the following functions:
1. `get_frequent_itemsets(transactions, min_support)` - Returns frequent itemsets and their support
2. `generate_candidates(prev_frequent_itemsets, k)` - Generates candidate itemsets of length `k`
3. `calculate_support(transactions, candidates)` - Calculates the support count for each candidate

**Write reusable functions** for each part of the algorithm.

In [ ]:
def get_frequent_itemsets(transactions, min_support):
    #This list will contain all the frequent itemsets
    frequentItemSets = []

    #This list will contain all the non-frequent itemsets which will be used for pruning
    nonFrequentItemSets = []

    # This format is used to create a new entry in the Frequent Item df 
    ItemSetFormat = {
        'ItemSet' : [],
        'SupportCount': []
    }
    candidateSet = transactions.sum().reset_index()
    candidateSet.columns = ['ItemSet', 'SupportCount']
    candidateSet['ItemSet'] = candidateSet['ItemSet'].map( lambda x : {x} )
    frequentItemSets.append(pd.DataFrame(ItemSetFormat))
    currentItemSets = frequentItemSets[0]
    for i in range(0, len(candidateSet)):
        itemSet = candidateSet.iloc[i]
        if itemSet.SupportCount >= min_support:
            currentItemSets.loc[len(currentItemSets)] = itemSet
        else:
            print('nonFreqItemSet:', itemSet['ItemSet'])
            nonFrequentItemSets.append(itemSet['ItemSet'])
    
    flag = True
    itemSetSize = 2
    while(flag):
        frequentItemSets.append(pd.DataFrame(ItemSetFormat))
        itemSet = generate_candidates(frequentItemSets[itemSetSize-2]['ItemSet'], itemSetSize)
        currentItemSets = frequentItemSets[itemSetSize-1]

        for s in itemSet:
            isFrequent = True
            for nonFrequentItemSet in nonFrequentItemSets:
                if nonFrequentItemSet.issubset(s):
                    isFrequent = False
                    nonFrequentItemSets.append(s)
                    break

            if isFrequent:
                supportCount = calculate_support(transactions, s)
                if supportCount >= min_support:
                    currentItemSets.loc[len(currentItemSets)] = [s, supportCount]
        itemSetSize += 1
        if len(currentItemSets) <= 1:
            return currentItemSets
    
def generate_candidates(prev_frequent_itemsets, k):
    candidateSets = []
    for s1 in prev_frequent_itemsets:
        for s2 in prev_frequent_itemsets:
            s = s1.union(s2)
            if len(s) == k and s not in candidateSets:
                candidateSets.append(s)
    return candidateSets

def calculate_support(transactions, candidates):
    candidates = list(candidates)
    column = transactions[candidates[0]]
    for i in range(1, len(candidates)):
        column = column & transactions[candidates[i]]
    return column.sum()


In [25]:
get_frequent_itemsets(basket, 1000)

nonFreqItemSet: {' 4 PURPLE FLOCK DINNER CANDLES'}
nonFreqItemSet: {" 50'S CHRISTMAS GIFT BAG LARGE"}
nonFreqItemSet: {' DOLLY GIRL BEAKER'}
nonFreqItemSet: {' I LOVE LONDON MINI BACKPACK'}
nonFreqItemSet: {' NINE DRAWER OFFICE TIDY'}
nonFreqItemSet: {' OVAL WALL MIRROR DIAMANTE '}
nonFreqItemSet: {' RED SPOT GIFT BAG LARGE'}
nonFreqItemSet: {' SET 2 TEA TOWELS I LOVE LONDON '}
nonFreqItemSet: {' SPACEBOY BABY GIFT SET'}
nonFreqItemSet: {' TOADSTOOL BEDSIDE LIGHT '}
nonFreqItemSet: {' TRELLIS COAT RACK'}
nonFreqItemSet: {'10 COLOUR SPACEBOY PEN'}
nonFreqItemSet: {'12 COLOURED PARTY BALLOONS'}
nonFreqItemSet: {'12 DAISY PEGS IN WOOD BOX'}
nonFreqItemSet: {'12 EGG HOUSE PAINTED WOOD'}
nonFreqItemSet: {'12 HANGING EGGS HAND PAINTED'}
nonFreqItemSet: {'12 IVORY ROSE PEG PLACE SETTINGS'}
nonFreqItemSet: {'12 MESSAGE CARDS WITH ENVELOPES'}
nonFreqItemSet: {'12 PENCIL SMALL TUBE WOODLAND'}
nonFreqItemSet: {'12 PENCILS SMALL TUBE RED RETROSPOT'}
nonFreqItemSet: {'12 PENCILS SMALL TUBE SKULL'}


,ItemSet,SupportCount


## Step 3: Generate Association Rules

- Use frequent itemsets to generate association rules
- For each rule `A => B`, calculate:
  - **Support**
  - **Confidence**
- Only return rules that meet a minimum confidence threshold (e.g., 0.5)

👉 **Implement rule generation function below**

In [27]:
from itertools import combinations

# Function to generate rules from frequent itemsets
def generate_rules(frequent_itemsets, min_confidence=0.5):
    rules = []
    for i in range(len(frequent_itemsets)):
        df = frequent_itemsets[i]
        for _, row in df.iterrows():
            itemset = row['ItemSet']
            support = row['SupportCount']
            
            if len(itemset) > 1:  # Only itemsets with 2+ items can form rules
                for j in range(1, len(itemset)):
                    for antecedent in combinations(itemset, j):
                        antecedent = frozenset(antecedent)
                        consequent = itemset - antecedent
                        
                        # Find support of antecedent
                        antecedent_support = None
                        for k in range(len(frequent_itemsets)):
                            temp_df = frequent_itemsets[k]
                            match = temp_df[temp_df['ItemSet'] == set(antecedent)]
                            if not match.empty:
                                antecedent_support = match.iloc[0]['SupportCount']
                                break
                        
                        if antecedent_support:  
                            confidence = support / antecedent_support
                            if confidence >= min_confidence:
                                rules.append({
                                    "antecedent": set(antecedent),
                                    "consequent": set(consequent),
                                    "support": round(support, 3),
                                    "confidence": round(confidence, 3)
                                })
    return pd.DataFrame(rules)


## Step 4: Output and Visualize

- Print top 10 frequent itemsets
- Print top 10 association rules (by confidence or lift)

👉 **Output results below**

In [ ]:
# Get frequent itemsets (you can change min_support as needed)
frequent_itemsets = get_frequent_itemsets(basket, min_support=30)

# Print Top 10 Frequent Itemsets
print("Top 10 Frequent Itemsets:")
print(frequent_itemsets.sort_values(by="SupportCount", ascending=False).head(10))

# Generate Rules
rules_df = generate_rules([frequent_itemsets], min_confidence=0.5)

# Print Top 10 Rules by Confidence
print("\nTop 10 Association Rules (by Confidence):")
print(rules_df.sort_values(by="confidence", ascending=False).head(10))


nonFreqItemSet: {' NINE DRAWER OFFICE TIDY'}
nonFreqItemSet: {' TOADSTOOL BEDSIDE LIGHT '}
nonFreqItemSet: {'12 HANGING EGGS HAND PAINTED'}
nonFreqItemSet: {'12 PINK HEN+CHICKS IN BASKET'}
nonFreqItemSet: {'12 PINK ROSE PEG PLACE SETTINGS'}
nonFreqItemSet: {'15 PINK FLUFFY CHICKS IN BOX'}
nonFreqItemSet: {'16 PC CUTLERY SET PANTRY DESIGN'}
nonFreqItemSet: {'18PC WOODEN CUTLERY SET DISPOSABLE'}
nonFreqItemSet: {'2 PICTURE BOOK EGGS EASTER BUNNY'}
nonFreqItemSet: {'2 PICTURE BOOK EGGS EASTER CHICKS'}
nonFreqItemSet: {'2 PICTURE BOOK EGGS EASTER DUCKS'}
nonFreqItemSet: {'3 BIRDS CANVAS SCREEN'}
nonFreqItemSet: {'3 BLACK CATS W HEARTS BLANK CARD'}
nonFreqItemSet: {'3 PINK HEN+CHICKS IN BASKET'}
nonFreqItemSet: {'3 TRADITIONAL COOKIE CUTTERS  SET'}
nonFreqItemSet: {'3 WICK CHRISTMAS BRIAR CANDLE '}
nonFreqItemSet: {'36 PENCILS TUBE POSY'}
nonFreqItemSet: {'3D HEARTS  HONEYCOMB PAPER GARLAND'}
nonFreqItemSet: {'4 BLUE DINNER CANDLES SILVER FLOCK'}
nonFreqItemSet: {'4 BURGUNDY WINE DINNER CAN